In [60]:
import requests
import json
import os

from kl_assets.api.espn import get_endpoint


leagues = {
    "nhl": "hockey/nhl",
    "nfl": "football/nfl",
    "nba": "basketball/nba",
    "mlb": "baseball/mlb",
}

for league, api_path in leagues.items():

    print(f"Processing {league.upper()}...")

    response = get_endpoint(f"sports/{api_path}/teams")
    data = response.json()

    if not os.path.exists(f"graphics/logos/{league}"):
        os.makedirs(f"graphics/logos/{league}")

    with open(f"utils/abbr_{league}.json", "r") as f:
        abbr_dict = json.load(f)

    for team in data["sports"][0]["leagues"][0]["teams"]:
        
        tm_data = team["team"]
        logo_url = tm_data["logos"][0]["href"]
        team_name = tm_data["displayName"]
        tm = abbr_dict[team_name]
        tm_id = tm_data["id"]

        with open(f"graphics/logos/{league}/{tm.upper()}.png", "wb") as f:
            img_resp = requests.get(logo_url)
            f.write(img_resp.content)

        roster_response = get_endpoint(f"sports/{api_path}/teams/{tm_id}/roster")


        if not os.path.exists(f"graphics/faces/{league}/{tm.upper()}"):
            os.makedirs(f"graphics/faces/{league}/{tm.upper()}")

        for athlete in roster_response.json()["athletes"]:
            player_id = athlete["id"]
            player_name = athlete["headshot"]["alt"]
            headshot_url = athlete["headshot"]["href"]
            sanitized_name = sanitize_name(player_name)

            with open(f"graphics/faces/{league}/{tm.upper()}/{player_id}_{sanitized_name}.png", "wb") as f:
                hs_response = requests.get(headshot_url)
                f.write(hs_response.content)
        break

    break

Processing NHL...


KeyError: 'id'

In [61]:
roster_response.json()["athletes"]

[{'position': 'Centers',
  'items': [{'id': '5149153',
    'uid': 's:70~l:90~a:5149153',
    'guid': '82d801a2-193c-3827-8822-1ebcd7d61dd2',
    'alternateIds': {'sdr': '5149153'},
    'alternateId': '5149153',
    'firstName': 'Leo',
    'lastName': 'Carlsson',
    'fullName': 'Leo Carlsson',
    'displayName': 'Leo Carlsson',
    'shortName': 'L. Carlsson',
    'weight': 203.0,
    'displayWeight': '203 lbs',
    'height': 75.0,
    'displayHeight': '6\' 3"',
    'age': 21,
    'dateOfBirth': '2004-12-26T08:00Z',
    'links': [{'language': 'en-US',
      'rel': ['playercard', 'desktop', 'athlete'],
      'href': 'https://www.espn.com/nhl/player/_/id/5149153',
      'text': 'Player Card',
      'shortText': 'Player Card',
      'isExternal': False,
      'isPremium': False},
     {'language': 'en-US',
      'rel': ['stats', 'desktop', 'athlete'],
      'href': 'https://www.espn.com/nhl/player/stats/_/id/5149153/leo-carlsson',
      'text': 'Stats',
      'shortText': 'Stats',
      'i

In [30]:
import re
import unicodedata

def sanitize_name(name: str) -> str:
    name = unicodedata.normalize('NFD', name)
    name = name.encode('ascii', 'ignore').decode('ascii')
    name = name.lower()
    name = re.sub(r'[^a-z]+', '_', name)
    name = name.strip('_')
    return name

In [31]:
sanitize_name(player_name)

'keaton_wallace'

In [39]:
r.json()["athletes"][3]["id"]

'5175647'